<a href="https://colab.research.google.com/github/bokofod/RothC_SOC/blob/main/download_gdrive_folder_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download a Google Drive folder (individual files, no zip)

Recursively downloads everything under a folder ID into `/content/...` on the Colab VM. Uses the **Drive API** (not “Download folder” as a zip). Sign in with an account that can open the folder in Drive.

**Folder:** [link](https://drive.google.com/drive/folders/1EsC6a156jS_wyB7f_l3qp-WySbKpw_Ty) — ID is set in the config cell below.

## 1. Dependencies

In [8]:
# Colab usually has google-api-python-client; this is safe if already installed
%pip install -q google-api-python-client google-auth-httplib2

## 2. Sign in to Google (Drive read access)

Run the cell; complete the browser popup. Your notebook session can then list and download files you are allowed to see.

In [9]:
from google.colab import auth

auth.authenticate_user()

import google.auth
import httplib2
import google_auth_httplib2
from googleapiclient.discovery import build

SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
creds, _ = google.auth.default(scopes=SCOPES)

# Explicit httplib2 timeout silences: "httplib2 transport does not support per-request timeout"
_http = httplib2.Http(timeout=300)
authorized_http = google_auth_httplib2.AuthorizedHttp(creds, http=_http)
service = build("drive", "v3", http=authorized_http, cache_discovery=False)

## 3. Config — edit folder ID and output path

In [10]:
from pathlib import Path

# From URL: https://drive.google.com/drive/folders/<THIS_PART>
# https://drive.google.com/drive/u/0/folders/1EsC6a156jS_wyB7f_l3qp-WySbKpw_Ty
FOLDER_ID = "1EsC6a156jS_wyB7f_l3qp-WySbKpw_Ty"

# Where to write files on the Colab VM (fast local disk)
OUTPUT_DIR = Path("/content/gdrive_geotiffs")

# Skip native Google Docs/Sheets/Slides (not raw GeoTIFFs)
SKIP_GOOGLE_NATIVE_FILES = True

# --- Filters (all optional; leave empty/None to disable) ------------------
# Allowed extensions, case-insensitive. Leading dot is optional.
# Empty list = accept any extension.
FILE_EXTENSIONS = [".tif", ".tiff", "tfw"]           # e.g. [".tif", ".tiff", "tfw"]

# Only download files modified STRICTLY AFTER this moment.
# "YYYY-MM-DD" (assumed UTC midnight) or full RFC 3339 like
# "2024-06-15T09:30:00Z". None = no date filter.
MODIFIED_AFTER = "2026-05-27" #None          # e.g. "2024-06-15"

# Name must contain at least ONE of these substrings (case-insensitive).
# Empty list = no substring filter.
NAME_INCLUDES = ["Mongolia"]             # e.g. ["eligible", "Mongolia"]

# Name must contain NONE of these substrings. Applied after INCLUDES, so
# excludes always win over includes.
NAME_EXCLUDES = []                # e.g. ["draft", "_old", "tmp"]

# When True, NAME_INCLUDES and NAME_EXCLUDES are compared case-sensitively.
# (Extension matching is always case-insensitive — filesystem convention.)
NAME_CASE_SENSITIVE = False


# Examples:
# Only GeoTIFFs (and their world files) from the Mongolia subset, but skip any draft/tmp/old files
# modified this year.
# FILE_EXTENSIONS = [".tif", ".tiff", ".tfw"]
# MODIFIED_AFTER  = "2026-01-01"
# NAME_INCLUDES   = ["Mongolia"]
# NAME_EXCLUDES       = ["draft", "tmp", "_old", "backup", "outdated"]
# NAME_CASE_SENSITIVE = False

# Match "Aimag" exactly (capital A), reject if name also contains "TEST"
# FILE_EXTENSIONS     = []
# MODIFIED_AFTER      = None
# NAME_INCLUDES       = ["Aimag"]
# NAME_EXCLUDES       = ["TEST"]
# NAME_CASE_SENSITIVE = True       # "aimag" / "AIMAG" would now NOT match

# Anything containing "eligible" OR "suitability" in the name, any type, any date
# FILE_EXTENSIONS = []
# MODIFIED_AFTER  = None
# NAME_INCLUDES   = ["eligible", "suitability"]

# Everything modified after a specific timestamp (UTC), no other filters
# FILE_EXTENSIONS = []
# MODIFIED_AFTER  = "2025-12-15T09:00:00Z"
# NAME_INCLUDES   = []

## 4. Download helpers (recursive, streams large files to disk)

In [11]:
import sys
from datetime import datetime, timezone
from googleapiclient.http import MediaIoBaseDownload

GFOLDER = "application/vnd.google-apps.folder"


def _normalize_extensions(exts):
    out = set()
    for e in exts or []:
        e = e.strip().lower()
        if not e:
            continue
        out.add(e if e.startswith(".") else f".{e}")
    return out


def _parse_after(ts):
    if not ts:
        return None
    s = ts.strip()
    if len(s) == 10:                       # "YYYY-MM-DD"
        s = s + "T00:00:00+00:00"
    s = s.replace("Z", "+00:00")           # Drive returns ...Z
    return datetime.fromisoformat(s).astimezone(timezone.utc)


def _file_passes_filters(name, modified_time_iso):
    """Re-reads filter globals on every call so editing Cell 3 and
    re-running Cell 5 takes effect without rerunning Cell 4."""
    cs = bool(NAME_CASE_SENSITIVE)
    haystack = name if cs else name.lower()

    # 1. Extension — always case-insensitive
    exts = _normalize_extensions(FILE_EXTENSIONS)
    if exts and not any(name.lower().endswith(e) for e in exts):
        return False

    # 2. Includes — at least one must appear (case per NAME_CASE_SENSITIVE)
    needles_in = [s if cs else s.lower()
                  for s in (NAME_INCLUDES or []) if s.strip()]
    if needles_in and not any(s in haystack for s in needles_in):
        return False

    # 3. Excludes — none may appear; excludes win over includes
    needles_ex = [s if cs else s.lower()
                  for s in (NAME_EXCLUDES or []) if s.strip()]
    if needles_ex and any(s in haystack for s in needles_ex):
        return False

    # 4. Modified-after — strictly after
    after = _parse_after(MODIFIED_AFTER)
    if after is not None:
        if not modified_time_iso:
            return False
        mt = datetime.fromisoformat(
            modified_time_iso.replace("Z", "+00:00")).astimezone(timezone.utc)
        if mt <= after:
            return False

    return True


def list_children(service, folder_id: str):
    page_token = None
    q = f"'{folder_id}' in parents and trashed = false"
    while True:
        resp = (
            service.files()
            .list(
                q=q,
                spaces="drive",
                fields="nextPageToken, files(id, name, mimeType, modifiedTime)",
                pageToken=page_token,
                pageSize=1000,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
            )
            .execute()
        )
        for f in resp.get("files", []):
            yield f["id"], f["name"], f["mimeType"], f.get("modifiedTime")
        page_token = resp.get("nextPageToken")
        if not page_token:
            break


def download_file(service, file_id: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
    with open(dest, "wb") as out:
        downloader = MediaIoBaseDownload(out, request)
        done = False
        while not done:
            _status, done = downloader.next_chunk()


def sync_folder(service, folder_id: str, dest_dir: Path,
                skip_google_native: bool) -> None:
    n_downloaded = 0
    n_skipped_native = 0
    n_skipped_filter = 0
    for fid, name, mime, mtime in list_children(service, folder_id):
        path = dest_dir / name
        if mime == GFOLDER:
            # Always recurse — filters apply to leaf files, not folders.
            sync_folder(service, fid, path, skip_google_native)
            continue
        if mime.startswith("application/vnd.google-apps.") and skip_google_native:
            print(f"skip (Google app file): {path}", file=sys.stderr)
            n_skipped_native += 1
            continue
        if not _file_passes_filters(name, mtime):
            print(f"skip (filter):          {path}", file=sys.stderr)
            n_skipped_filter += 1
            continue
        print(path)
        download_file(service, fid, path)
        n_downloaded += 1
    if dest_dir == OUTPUT_DIR:  # top-level summary only
        print(
            f"\nSummary: {n_downloaded} downloaded, "
            f"{n_skipped_filter} skipped by filter, "
            f"{n_skipped_native} skipped Google-native"
        )

## 5. Run download

In [12]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Echo active filters so the run header is self-documenting.
print(f"Folder ID:       {FOLDER_ID}")
print(f"Output dir:      {OUTPUT_DIR}")
print(f"Extensions:      {FILE_EXTENSIONS or 'any'}")
print(f"Modified after:  {MODIFIED_AFTER or 'no date filter'}")
print(f"Name includes:   {NAME_INCLUDES or 'no substring filter'}")
print(f"Name excludes:      {NAME_EXCLUDES or 'no exclude filter'}")
print(f"Case sensitive:     {NAME_CASE_SENSITIVE}")
print(f"Skip Google-native: {SKIP_GOOGLE_NATIVE_FILES}")
print("-" * 60)

sync_folder(service, FOLDER_ID, OUTPUT_DIR, SKIP_GOOGLE_NATIVE_FILES)
print("Done. Files under:", OUTPUT_DIR)

Folder ID:       1EsC6a156jS_wyB7f_l3qp-WySbKpw_Ty
Output dir:      /content/gdrive_geotiffs
Extensions:      ['.tif', '.tiff', 'tfw']
Modified after:  2026-05-27
Name includes:   ['Mongolia']
Name excludes:      no exclude filter
Case sensitive:     False
Skip Google-native: True
------------------------------------------------------------
/content/gdrive_geotiffs/LimitingFactor_Mongolia_Ulmus_macrocarpa_29May26_1135_54-0000000000-0000065536.tif
/content/gdrive_geotiffs/LimitingFactor_Mongolia_Ulmus_macrocarpa_29May26_1135_54-0000000000-0000000000.tif
/content/gdrive_geotiffs/FailureCount_Mongolia_Abies_sibirica_29May26_1126_20-0000000000-0000065536.tif
/content/gdrive_geotiffs/FailureCount_Mongolia_Abies_sibirica_29May26_1126_20-0000000000-0000000000.tif
/content/gdrive_geotiffs/LimitingFactor_Mongolia_Tamarix_ramosissima_29May26_1135_54-0000000000-0000065536.tif
/content/gdrive_geotiffs/LimitingFactor_Mongolia_Tamarix_ramosissima_29May26_1135_54-0000000000-0000000000.tif
/content/gd

/content/gdrive_geotiffs/FailureCount_Mongolia_Populus_sibirica_29May26_1126_20-0000000000-0000065536.tif


/content/gdrive_geotiffs/FailureCount_Mongolia_Populus_sibirica_29May26_1126_20-0000000000-0000000000.tif
/content/gdrive_geotiffs/FailureCount_Mongolia_Pinus_sylvestris_29May26_1126_20-0000000000-0000065536.tif
/content/gdrive_geotiffs/FailureCount_Mongolia_Pinus_sylvestris_29May26_1126_20-0000000000-0000000000.tif
/content/gdrive_geotiffs/LimitingFactor_Mongolia_Populus_laurifolia_29May26_1126_20-0000000000-0000065536.tif
/content/gdrive_geotiffs/LimitingFactor_Mongolia_Populus_laurifolia_29May26_1126_20-0000000000-0000000000.tif
/content/gdrive_geotiffs/Suitability_Mongolia_Populus_euphratica_Broadleaf_AllParams_29May26_1126_20-0000000000-0000065536.tif
/content/gdrive_geotiffs/Suitability_Mongolia_Populus_euphratica_Broadleaf_AllParams_29May26_1126_20-0000000000-0000000000.tif
/content/gdrive_geotiffs/Suitability_Mongolia_Populus_sibirica_Broadleaf_AllParams_29May26_1126_20-0000000000-0000065536.tif
/content/gdrive_geotiffs/Suitability_Mongolia_Populus_sibirica_Broadleaf_AllParams_

skip (filter):          /content/gdrive_geotiffs/FailureCount_Theobroma_cacao_12Feb26_1031_51.tif
skip (filter):          /content/gdrive_geotiffs/FailureCount_Terminalia_myriocarpa_12Feb26_1031_51.tif
skip (filter):          /content/gdrive_geotiffs/LimitingFactor_Terminalia_myriocarpa_12Feb26_1031_51.tif
skip (filter):          /content/gdrive_geotiffs/FailureCount_Psidium_guajava_12Feb26_1031_51.tif
skip (filter):          /content/gdrive_geotiffs/LimitingFactor_Theobroma_cacao_12Feb26_1031_51.tif
skip (filter):          /content/gdrive_geotiffs/Suitability_Theobroma_cacao_Unknown_AllParams_Spec_12Feb26_1031_51.tif
skip (filter):          /content/gdrive_geotiffs/Suitability_Terminalia_myriocarpa_Unknown_AllParams_12Feb26_1031_51.tif
skip (filter):          /content/gdrive_geotiffs/Suitability_Swietenia_mahagoni_Unknown_AllParams_Gen_12Feb26_1031_51.tif
skip (filter):          /content/gdrive_geotiffs/LimitingFactor_Prunus_serrulata_12Feb26_1031_51.tif
skip (filter):          /conte

## 6. (Optional) Zip or copy to your Drive

- Zip the folder if you want a single archive to download from Colab’s file browser.
- Or mount Drive and `shutil.copytree` into `MyDrive`.

In [13]:
# Optional: zip /content/gdrive_geotiffs → /content/geotiffs.zip for one-click download
import shutil
shutil.make_archive("/content/geotiffs", "zip", OUTPUT_DIR)

# Optional: copy to Google Drive (uncomment)
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copytree(OUTPUT_DIR, "/content/drive/MyDrive/gdrive_geotiffs", dirs_exist_ok=True)


# AFTER THIS CELL FINISHED - you have two artifacts on the Colab VM:

# /content/geotiffs.zip — the archive
# /content/gdrive_geotiffs/*.tif — the unzipped folder
# Easiest: file browser
# In Colab's left sidebar, click the 📁 folder icon ("Files").
# You'll see /content/ listed. Scroll until you see geotiffs.zip.
# Right-click geotiffs.zip → Download. Your browser saves it to C:\Users\bokof\Downloads\ (or wherever your browser is configured).
# If the file isn't visible, click the refresh ↻ button at the top of the file panel.

# Alternative: trigger the download from Python
# Paste this in a new cell and run it — your browser will start the download as soon as the cell finishes:

'/content/geotiffs.zip'